In [ ]:
from pathlib import Path
import sys

import geopandas as gpd
import numpy as np
from osgeo import gdal
import pandas as pd
from transliterate import translit

sys.path.append("../src")

from data_processing.gdal_processing import (
    create_mosaic,
    flood_extent_tiles,
    gdal_extent_clipper,
)
from utils.logger import setup_logger

log = setup_logger("ZenodoDataset", log_file="../logs/zenodo_dataset.log")
save_path = Path("../data/zenodo")
save_path.mkdir(parents=True, exist_ok=True)


In [ ]:
len(list(Path("/home/dmbrmv/Development/camels_ru/data/Russia/HydroFiles/Discharge").glob("**/*.csv")))

In [ ]:
roi_watersheds = Path("/media/dmbrmv/ssd_2tb/CAMELS_RU/geometry_v2/temp/roi_22/").glob("roi_22_watershed_*.gpkg")
watersheds_concat = pd.concat([gpd.read_file(f) for f in roi_watersheds])
watersheds_concat.set_index("gauge_id", inplace=True)
watersheds_concat = gpd.GeoDataFrame(watersheds_concat)
watersheds_concat.crs = "EPSG:4326"
watersheds_concat.to_file("/media/dmbrmv/ssd_2tb/CAMELS_RU/geometry_v2/temp/roi_22/roi_22_watersheds.gpkg")

In [ ]:
roi_watersheds = Path("/media/dmbrmv/ssd_2tb/CAMELS_RU/geometry_v2/temp/roi_24/").glob("roi_24_watershed_*.gpkg")
watersheds_concat = pd.concat([gpd.read_file(f) for f in roi_watersheds])
watersheds_concat.set_index("gauge_id", inplace=True)
watersheds_concat = gpd.GeoDataFrame(watersheds_concat)
watersheds_concat.crs = "EPSG:4326"
watersheds_concat.to_file("/media/dmbrmv/ssd_2tb/CAMELS_RU/geometry_v2/temp/roi_24/roi_24_watersheds.gpkg")

# Proper roi`s

In [ ]:
import pyogrio
roi_ids = [10, 12, 13, 14, 16, 18, 19, 20, 21, 22, 24, 28]


def parse_area_value(value):
    """Parse area values, handling fractions/ranges like '20800/22100'."""
    if pd.isna(value):
        return np.nan

    # Convert to string and strip whitespace
    s = str(value).strip()

    # Replace comma with dot for decimal separator
    s = s.replace(",", ".")

    # Handle fraction/range values like '20800/22100'
    if "/" in s:
        parts = s.split("/")
        if len(parts) == 2:
            try:
                # Try to parse both parts and take average
                val1 = float(parts[0].strip())
                val2 = float(parts[1].strip())
                return (val1 + val2) / 2.0
            except (ValueError, AttributeError):
                pass

    # Try direct conversion
    try:
        return float(s)
    except (ValueError, AttributeError):
        # If all else fails, return NaN
        return np.nan


roi_watersheds = {
    roi: Path(f"/media/dmbrmv/ssd_2tb/CAMELS_RU/geometry_v2/temp/roi_{roi}/roi_{roi}_watersheds.gpkg") for roi in roi_ids
}

rshdmt_stats = pd.read_csv(
    "/media/dmbrmv/ssd_2tb/CAMELS_RU/hydropost_catalog.csv",
)

rshdmt_stats.rename(columns={"Unnamed: 0": "gauge_id", "Unnamed: 7": "area_roshydromet"}, inplace=True)

rshdmt_stats = rshdmt_stats[["gauge_id", "area_roshydromet"]]

rshdmt_stats["gauge_id"] = rshdmt_stats["gauge_id"].astype(str)

# Apply parsing function to convert area values
parsed_areas = [parse_area_value(val) for val in rshdmt_stats["area_roshydromet"]]
rshdmt_stats["area_roshydromet"] = parsed_areas
rshdmt_stats.set_index("gauge_id", inplace=True)


def _read_ws(ws_path: Path) -> gpd.GeoDataFrame:
    ws = gpd.read_file(ws_path)
    ws.set_index("gauge_id", inplace=True)

    return ws


for _roi_id in roi_ids:
    try:
        _ws = _read_ws(roi_watersheds[_roi_id])
    except pyogrio.errors.DataSourceError:
        continue
    for gauge_id in _ws.index:
        try:
            _ws.loc[gauge_id, "area_roshydromet"] = rshdmt_stats.loc[gauge_id, "area_roshydromet"]
            if rshdmt_stats.loc[gauge_id, "area_roshydromet"] > 0:
                _ws.loc[gauge_id, "area_diff_perc"] = (
                    (rshdmt_stats.loc[gauge_id, "area_roshydromet"] - _ws.loc[gauge_id, "area_km2"])
                    / rshdmt_stats.loc[gauge_id, "area_roshydromet"]
                    * 100
                )
            else:
                _ws.loc[gauge_id, "area_diff_perc"] = np.nan
        except KeyError:
            _ws.loc[gauge_id, "area_roshydromet"] = np.nan
            _ws.loc[gauge_id, "area_diff_perc"] = np.nan
    _ws = _ws[["name", "area_km2", "area_roshydromet", "area_diff_perc", "geometry"]]
    _ws.to_file(roi_watersheds[_roi_id], driver="GPKG")

In [ ]:
roi_watersheds = Path("/media/dmbrmv/ssd_2tb/CAMELS_RU/geometry_v2/temp/").glob("roi_*/roi_*_watersheds.gpkg")
watersheds_concat = pd.concat([gpd.read_file(f) for f in roi_watersheds])
watersheds_concat.set_index("gauge_id", inplace=True)
watersheds_concat = gpd.GeoDataFrame(watersheds_concat)
watersheds_concat.crs = "EPSG:4326"
watersheds_concat.to_file("/media/dmbrmv/ssd_2tb/CAMELS_RU/geometry/camels_watersheds.gpkg")

roi_gauges = Path("/media/dmbrmv/ssd_2tb/CAMELS_RU/geometry_v2/temp/").glob("roi_*/roi_*_gauges.gpkg")
gauges_concat = pd.concat([gpd.read_file(f) for f in roi_gauges])
gauges_concat.set_index("gauge_id", inplace=True)
gauges_concat = gpd.GeoDataFrame(gauges_concat)
gauges_concat.crs = "EPSG:4326"
gauges_concat.loc[:, "area_diff_perc"] = watersheds_concat.loc[gauges_concat.index, "area_diff_perc"].astype(float)
gauges_concat.loc[:, "area_roshydromet"] = watersheds_concat.loc[gauges_concat.index, "area_roshydromet"].astype(float)
gauges_concat.to_file("/media/dmbrmv/ssd_2tb/CAMELS_RU/geometry/camels_gauges.gpkg")

In [ ]:
attr_path = save_path / "attributes"
attr_path.mkdir(parents=True, exist_ok=True)
pd.read_csv(
    "/home/dmbrmv/Development/camels_ru/data/attributes/hydro_atlas_cis_camels.csv",
    index_col="gauge_id",
    dtype={"gauge_id": str},
).to_csv(attr_path / "camels_ru_attributes.csv")

In [ ]:
geom_path = save_path / "geometry"
geom_path.mkdir(parents=True, exist_ok=True)

ws_file = gpd.read_file("../data/Geometry/WatershedGeomCAMELS.gpkg")
ws_file.set_index("gauge_id", inplace=True)

ws_with_diff = gpd.read_file("../data/Geometry/russia_full.gpkg")
ws_with_diff.set_index("gauge_id", inplace=True)

ws_file.insert(3, "area_roshydromet", value=np.nan)
ws_file.insert(3, "area_diff", value=np.nan)
full_gauges = gpd.read_file(geom_path / "camels_ru_gauges.gpkg")

full_gauges.set_index("gauge_id", inplace=True)

In [ ]:
old_gauges = gpd.read_file("../data/Geometry/2803_gauge.gpkg")

old_gauges.set_index("gauge_id", inplace=True)

In [ ]:
# Step 1: Overwrite geometry in full_gauges with old_gauges geometry
common_gauges = full_gauges.index.intersection(old_gauges.index)
print(f"Found {len(common_gauges)} common gauges between full_gauges and old_gauges")

# Update geometry for common gauges
for gauge_id in common_gauges:
    full_gauges.loc[gauge_id, "geometry"] = old_gauges.loc[gauge_id, "geometry"]

print(f"✓ Overwrote geometry for {len(common_gauges)} gauges")

# Step 2: Find gauges that are only in old_gauges (not in full_gauges)
only_in_old = old_gauges.index.difference(full_gauges.index)
only_in_full = full_gauges.index.difference(old_gauges.index)
print(f"\nFound {len(only_in_old)} gauges only in old_gauges")
print(f"Found {len(only_in_full)} gauges only in full_gauges")

# Step 3: Merge datasets - add gauges from old_gauges that aren't in full_gauges
if len(only_in_old) > 0:
    # Create a subset of old_gauges with only the missing gauges
    gauges_to_add = old_gauges.loc[only_in_old].copy()

    # Ensure columns match - add missing columns with NaN
    for col in full_gauges.columns:
        if col not in gauges_to_add.columns and col != "geometry":
            gauges_to_add[col] = np.nan

    # Reorder columns to match full_gauges
    gauges_to_add = gauges_to_add[full_gauges.columns]

    # Concatenate the datasets (full_gauges already has updated geometry + unique gauges)
    final_gauges = pd.concat([full_gauges, gauges_to_add])

    print(f"✓ Added {len(only_in_old)} new gauges from old_gauges")
    print(f"  Example new gauge IDs: {list(only_in_old[:5])}")
else:
    final_gauges = full_gauges.copy()
    print("✓ No new gauges to add from old_gauges")

print(f"\n{'=' * 60}")
print("Final dataset composition:")
print(f"  - {len(common_gauges)} gauges with updated geometry")
print(f"  - {len(only_in_full)} gauges kept from full_gauges")
print(f"  - {len(only_in_old)} gauges added from old_gauges")
print(f"  = {len(final_gauges)} total gauges")
print(f"{'=' * 60}")

# Optional: Check for any additional columns in old_gauges
extra_cols = set(old_gauges.columns) - set(final_gauges.columns) - {"geometry"}
if extra_cols:
    print(f"\nNote: old_gauges has these extra columns not copied: {extra_cols}")
final_gauges.to_file("../data/Geometry/CompleteGauges2025.gpkg")


In [ ]:
# Get all gauge IDs from discharge files
discharge_decent = Path("../data/HydroFiles/Discharge/full/decent")
discharge_poor = Path("../data/HydroFiles/Discharge/full/poor")

gauge_ids_decent = set()
gauge_ids_poor = set()

if discharge_decent.exists():
    gauge_ids_decent = {f.stem for f in discharge_decent.glob("*.csv")}
    print(f"Found {len(gauge_ids_decent)} gauges in decent quality discharge data")

if discharge_poor.exists():
    gauge_ids_poor = {f.stem for f in discharge_poor.glob("*.csv")}
    print(f"Found {len(gauge_ids_poor)} gauges in poor quality discharge data")

all_discharge_gauges = gauge_ids_decent | gauge_ids_poor
print(f"Total unique gauge IDs in discharge data: {len(all_discharge_gauges)}")

# Get gauge IDs from full_gauges
gauges_in_dataset = set(full_gauges.index.astype(str))
print(f"Total gauge IDs in full_gauges: {len(gauges_in_dataset)}")

# Find missing gauges
missing_gauges = all_discharge_gauges - gauges_in_dataset
print(f"\n{'=' * 60}")
print(f"Missing {len(missing_gauges)} gauges from full_gauges:")
print(f"{'=' * 60}")

if missing_gauges:
    for gauge_id in sorted(missing_gauges):
        quality = "decent" if gauge_id in gauge_ids_decent else "poor"
        print(f"  {gauge_id} ({quality})")
else:
    print("✓ All discharge gauges are present in full_gauges!")


### Geometry

In [ ]:
full_gauges = gpd.read_file("../data/Geometry/GaugesFull.gpkg")
full_gauges["wmo_id"] = full_gauges["wmo_id"].astype(str)
full_gauges.set_index("wmo_id", inplace=True)
full_gauges.index.name = "gauge_id"
full_gauges = full_gauges[["name", "height", "area", "geometry"]]
full_gauges["area"] = full_gauges["area"].str.replace(r"[^\d.-]", "", regex=True)
full_gauges["area"] = pd.to_numeric(full_gauges["area"], errors="coerce")
full_gauges.replace({"area": {0.0: np.nan}}, inplace=True)
full_gauges["height"] = [h.replace(",", ".") if isinstance(h, str) else h for h in full_gauges["height"]]
full_gauges["height"] = full_gauges["height"].str.replace(r"[^\d.-]", "", regex=True)
full_gauges["height"] = pd.to_numeric(full_gauges["height"], errors="coerce")
full_gauges.rename(columns={"name": "name_ru"}, inplace=True)
full_gauges["name_en"] = [translit(n, "ru", reversed=True) for n in full_gauges["name_ru"]]

full_gauges = full_gauges[["name_en", "name_ru", "height", "area", "geometry"]]
full_gauges.to_file(geom_path / "camels_ru_gauges.gpkg")

#### Map Hybas with MeritHYDRO for Spatial coverage

In [ ]:
hybas_mapping = gpd.read_file("../data/Geometry/HybasSelection.gpkg")
hybas_mapping = hybas_mapping[["OBJECTID", "geometry"]]

name_dict = {
    "elv": "adjusted_elevation",
    "dir": "flow_direction",
    "upg": "upstream_drainage_pixels",
}

dtype_dict = {
    "dir": gdal.GDT_Byte,
    "elv": gdal.GDT_Float32,
    "upg": gdal.GDT_Int32,
}
nodata_dict = {"dir": 247, "elv": -9999.0, "upg": -9999}


elv_path = Path("../data/SpatialData/MeritRU")
elv_path.mkdir(parents=True, exist_ok=True)
tmp_tif = Path("../data/tmp/rasters")
tmp_tif.mkdir(parents=True, exist_ok=True)

for data_tag in ["elv", "dir", "upg"]:
    raster_path = elv_path / data_tag
    raster_path.mkdir(parents=True, exist_ok=True)
    for i, (obj_id, roi) in enumerate(hybas_mapping.itertuples(index=False)):
        hybas_mapping.iloc[[i], :].to_file(tmp_tif / f"{obj_id}_boundary.gpkg", driver="GPKG")
        # tif_epsg = hybas_mapping.iloc[[i], :].estimate_utm_crs().to_epsg()
        tif_epsg = 4326
        upper_left_x, lower_right_y, lower_right_x, upper_left_y = roi.bounds
        wgs_window = (upper_left_x, upper_left_y, lower_right_x, lower_right_y)
        tiles = flood_extent_tiles(
            topo_p="/mnt/storage/ResearchData/World/DEM/MeritDEM/",
            extent_coords=wgs_window,
            variable_dict=name_dict,
        )
        data_mosaic = create_mosaic(
            file_name=f"hybas_{obj_id}",
            file_path=elv_path,
            tiles=tiles[data_tag],
        )
        gdal_extent_clipper(
            initial_tif=data_mosaic,
            extent=wgs_window,
            tmp_tif=f"{tmp_tif}/{obj_id}_{data_tag}.tif",
            final_tif=f"{raster_path}/{obj_id}_{data_tag}.tif",
            crs_epsg=tif_epsg,
            dtype=dtype_dict[data_tag],
            nodata=nodata_dict[data_tag],
        )


In [ ]:
hybas_mapping = gpd.read_file("../data/Geometry/HybasSelection.gpkg")
hybas_mapping = hybas_mapping[["OBJECTID", "geometry"]]

obj_id = 28
roi = hybas_mapping.loc[hybas_mapping["OBJECTID"] == obj_id, "geometry"].values[0]

name_dict = {
    "elv": "adjusted_elevation",
    "dir": "flow_direction",
    "upg": "upstream_drainage_pixels",
}

dtype_dict = {
    "dir": gdal.GDT_Byte,
    "elv": gdal.GDT_Float32,
    "upg": gdal.GDT_Int32,
}
nodata_dict = {"dir": 247, "elv": -9999.0, "upg": -9999}


elv_path = Path("../data/SpatialData/MeritRU")
elv_path.mkdir(parents=True, exist_ok=True)
tmp_tif = Path("../data/tmp/rasters")
tmp_tif.mkdir(parents=True, exist_ok=True)

hybas_mapping.loc[hybas_mapping["OBJECTID"] == obj_id, :].to_file(
    tmp_tif / f"{obj_id}_boundary.gpkg", driver="GPKG"
)
# tif_epsg = hybas_mapping.iloc[[i], :].estimate_utm_crs().to_epsg()
tif_epsg = 4326
upper_left_x, lower_right_y, lower_right_x, upper_left_y = roi.bounds
wgs_window = (upper_left_x, upper_left_y, lower_right_x, lower_right_y)

for data_tag in ["elv", "dir", "upg"]:
    raster_path = elv_path / data_tag
    raster_path.mkdir(parents=True, exist_ok=True)
    tiles = flood_extent_tiles(
        topo_p="/mnt/storage/ResearchData/World/DEM/MeritDEM/",
        extent_coords=wgs_window,
        variable_dict=name_dict,
    )
    data_mosaic = create_mosaic(
        file_name=f"hybas_{obj_id}",
        file_path=elv_path,
        tiles=tiles[data_tag],
    )
    gdal_extent_clipper(
        initial_tif=data_mosaic,
        extent=wgs_window,
        tmp_tif=f"{tmp_tif}/{obj_id}_{data_tag}.tif",
        final_tif=f"{raster_path}/{obj_id}_{data_tag}.tif",
        crs_epsg=tif_epsg,
        dtype=dtype_dict[data_tag],
        nodata=nodata_dict[data_tag],
    )

In [ ]:
hydro_path = save_path / "hydrology"
hydro_path.mkdir(parents=True, exist_ok=True)


In [ ]:
meteo_path = save_path / "meteo"
meteo_path.mkdir(parents=True, exist_ok=True)

In [ ]:
raw_data = save_path / "raw"
raw_data.mkdir(parents=True, exist_ok=True)